# NumPy Dojo — Block 3: Linear Algebra + Block 4: Optimizers

Dataset: **word embeddings** — synthetic vectors of shape (V, D) where V=500 vocab, D=128 dims.  
Mimics working with real embedding matrices.

Also uses structured 2D point clouds for PCA.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


In [2]:

V, D = 500, 128
# Simulated word embeddings — normalized to unit sphere (common in practice)
embeddings_raw = np.random.randn(V, D)
embeddings = embeddings_raw / np.linalg.norm(embeddings_raw, axis=1, keepdims=True)


In [ ]:

# 2D structured dataset for PCA viz — two correlated Gaussians
mean1, cov1 = [2, 3], [[3, 2], [2, 2]]
mean2, cov2 = [-2, -1], [[2, -1], [-1, 1]]
X_2d = np.vstack([
    np.random.multivariate_normal(mean1, cov1, 200),
    np.random.multivariate_normal(mean2, cov2, 200)
])  # (400, 2)

# High-dimensional dataset for PCA reduction
# Low-rank structure: true data lives in 5D subspace embedded in 50D
true_components = np.random.randn(5, 50)
true_components /= np.linalg.norm(true_components, axis=1, keepdims=True)
latent = np.random.randn(300, 5)
X_hd = latent @ true_components + 0.1 * np.random.randn(300, 50)  # (300, 50)

print(f'embeddings:  {embeddings.shape}')
print(f'X_2d:        {X_2d.shape}')
print(f'X_hd:        {X_hd.shape}  (true rank ≈ 5)')

plt.figure(figsize=(5, 4))
plt.scatter(X_2d[:200, 0], X_2d[:200, 1], alpha=0.5, s=15, label='Cluster 1')
plt.scatter(X_2d[200:, 0], X_2d[200:, 1], alpha=0.5, s=15, label='Cluster 2')
plt.legend()
plt.title('X_2d — input to PCA')
plt.tight_layout()
plt.show()

---
## P8 — Pairwise Distance Matrix

Given `Q` (N, D) query embeddings and `K` (M, D) key embeddings, compute the full **(N, M)** pairwise L2 distance matrix — **no loops**.

Use the identity: `||x - y||² = ||x||² + ||y||² - 2 x·yᵀ`

Then:
- Find the nearest neighbor in K for each query in Q
- Find all pairs within distance threshold `t=0.8`

In [ ]:
Q = embeddings[:50]   # 50 query vectors
K = embeddings[50:]   # remaining as key vectors

def pairwise_l2(Q, K):
    """
    Q: (N, D), K: (M, D)
    Returns: dist (N, M) — Euclidean distances
    """
    # TODO: use the ||x-y||² = ||x||² + ||y||² - 2x·yᵀ identity
    # Use np.clip before sqrt to avoid tiny negatives from floating point
    pass

def nearest_neighbor(Q, K):
    """Returns indices into K for each query in Q. Shape: (N,)"""
    # TODO: use pairwise_l2, then argmin
    pass

def pairs_within_threshold(Q, K, t):
    """
    Returns (row_indices, col_indices) of all (Q_i, K_j) pairs
    where distance < t
    """
    # TODO: distance matrix → boolean mask → np.where
    pass

dist_matrix = pairwise_l2(Q, K)
nn_indices = nearest_neighbor(Q, K)
close_pairs = pairs_within_threshold(Q, K, t=0.8)

In [ ]:
# --- ASSERTS ---
N, M = len(Q), len(K)
assert dist_matrix.shape == (N, M), f'Expected ({N},{M}), got {dist_matrix.shape}'
assert np.all(dist_matrix >= 0), 'Distances must be non-negative'

# Self-distance check: distance from a vector to itself should be ~0
self_dist = pairwise_l2(Q[:5], Q[:5])
assert np.allclose(np.diag(self_dist), 0, atol=1e-5), 'Self-distance should be 0'

# Nearest neighbor: unit norm vectors — closest L2 ≈ highest cosine sim
# Verify nn_indices are valid indices into K
assert nn_indices.shape == (N,)
assert np.all(nn_indices >= 0) and np.all(nn_indices < M)

# Pair threshold
qi, ki = close_pairs
assert np.all(dist_matrix[qi, ki] < 0.8), 'Threshold filtering incorrect'

print(f'P8 PASSED ✓')
print(f'  Distance matrix: {dist_matrix.shape}')
print(f'  Pairs within t=0.8: {len(qi)}')

In [ ]:
# Visualize distance heatmap
plt.figure(figsize=(8, 6))
plt.imshow(dist_matrix[:20, :20], aspect='auto', cmap='viridis')
plt.colorbar(label='L2 Distance')
plt.title('Pairwise L2 Distance (first 20 queries × 20 keys)')
plt.xlabel('Key index')
plt.ylabel('Query index')
plt.tight_layout()
plt.show()

---
## P9 — PCA from Scratch

Implement PCA using eigendecomposition — **not SVD** (you'll see why the results should match).

Steps:
1. Center the data
2. Compute covariance matrix
3. Eigendecompose (you can use `np.linalg.eig`)
4. Sort by descending eigenvalue
5. Project onto top-k components
6. Compute explained variance ratio

Apply to both `X_2d` (k=2, visualize) and `X_hd` (k=5, check reconstruction).

In [ ]:
def pca(X, k):
    """
    X: (N, D)
    k: number of components to keep
    Returns:
        X_proj:    (N, k) — projected data
        components: (k, D) — top-k eigenvectors (row = component)
        explained_var_ratio: (k,) — fraction of variance explained by each component
        mean:      (D,) — for reconstruction
    """
    # Step 1: TODO — center

    # Step 2: TODO — covariance matrix (D, D)
    # Note: unbiased = divide by N-1

    # Step 3: TODO — eigendecompose
    # np.linalg.eig returns complex numbers sometimes — take .real

    # Step 4: TODO — sort by descending eigenvalue

    # Step 5: TODO — project centered X onto top-k eigenvectors

    # Step 6: TODO — explained variance ratio per component

    pass

def pca_reconstruct(X_proj, components, mean):
    """
    Project back to original space.
    X_proj: (N, k), components: (k, D), mean: (D,)
    Returns: (N, D)
    """
    # TODO
    pass

X_2d_proj, comps_2d, evr_2d, mean_2d = pca(X_2d, k=2)
X_hd_proj, comps_hd, evr_hd, mean_hd = pca(X_hd, k=5)

In [ ]:
# --- ASSERTS ---

# Shape checks
assert X_2d_proj.shape == (400, 2), f'2D proj shape wrong: {X_2d_proj.shape}'
assert X_hd_proj.shape == (300, 5), f'HD proj shape wrong: {X_hd_proj.shape}'

# Explained variance should be in (0, 1] and sum <= 1
assert np.all(evr_2d > 0)
assert evr_2d.sum() <= 1 + 1e-6

# For X_hd: top 5 components should explain most variance (data was constructed that way)
assert evr_hd.sum() > 0.85, f'Top 5 components should explain >85% variance, got {evr_hd.sum()*100:.1f}%'

# Reconstruction: full-rank PCA (k=D) should reconstruct exactly
N, D = X_2d.shape
X_proj_full, comps_full, _, mean_full = pca(X_2d, k=D)
X_recon = pca_reconstruct(X_proj_full, comps_full, mean_full)
assert np.allclose(X_recon, X_2d, atol=1e-6), 'Full-rank PCA should reconstruct exactly'

# Principal components should be orthonormal
gram = comps_2d @ comps_2d.T  # (k, k) should be identity
assert np.allclose(gram, np.eye(len(comps_2d)), atol=1e-5), 'Components not orthonormal'

print(f'P9 PASSED ✓')
print(f'  X_2d  explained variance: {evr_2d}')
print(f'  X_hd  top-5 explains:     {evr_hd.sum()*100:.1f}% of variance')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original data
axes[0].scatter(X_2d[:200, 0], X_2d[:200, 1], alpha=0.5, s=15, label='Cluster 1')
axes[0].scatter(X_2d[200:, 0], X_2d[200:, 1], alpha=0.5, s=15, label='Cluster 2')
# Draw principal components as arrows
scale = 3.0
for i, (c, ev) in enumerate(zip(comps_2d, evr_2d)):
    axes[0].annotate('', xy=mean_2d + scale * c, xytext=mean_2d,
                    arrowprops=dict(arrowstyle='->', color='red', lw=2))
    axes[0].text(*(mean_2d + scale * c * 1.1), f'PC{i+1} ({ev*100:.0f}%)', color='red', fontsize=9)
axes[0].set_title('Original data + Principal Components')
axes[0].legend()

# PCA projected data
axes[1].scatter(X_2d_proj[:200, 0], X_2d_proj[:200, 1], alpha=0.5, s=15, label='Cluster 1')
axes[1].scatter(X_2d_proj[200:, 0], X_2d_proj[200:, 1], alpha=0.5, s=15, label='Cluster 2')
axes[1].set_title('PCA Projected (PC1 vs PC2)')
axes[1].legend()

plt.tight_layout()
plt.show()

# Scree plot for X_hd
_, _, evr_all, _ = pca(X_hd, k=20)
plt.figure(figsize=(6, 4))
plt.bar(range(1, 21), evr_all * 100, color='steelblue')
plt.xlabel('Component')
plt.ylabel('% Variance Explained')
plt.title('Scree Plot — X_hd (true rank≈5)')
plt.tight_layout()
plt.show()

---
## P10 — Batch Matrix Operations

Batched linear algebra — common in attention mechanisms and batch processing.

Given:
- `A`: (B, N, M) — batch of matrices
- `B_mat`: (B, M, K) — batch of matrices

Implement:
1. Batch matmul → (B, N, K) using `np.einsum`
2. Batch trace of square (B, N, N) matrices — no loop
3. Batch Frobenius norm of (B, N, M) — no loop
4. Scaled dot-product attention scores: given Q (B, T, d_k), K (B, T, d_k), return (B, T, T) softmax'd weights

In [ ]:
B_size, N_size, M_size, K_size = 8, 10, 12, 6
A_batch = np.random.randn(B_size, N_size, M_size)
B_batch = np.random.randn(B_size, M_size, K_size)
S_batch = np.random.randn(B_size, N_size, N_size)  # square, for trace

T, d_k = 16, 32  # seq length, key dim
Q_attn = np.random.randn(B_size, T, d_k)
K_attn = np.random.randn(B_size, T, d_k)

def batch_matmul(A, B):
    """(B,N,M) x (B,M,K) -> (B,N,K) using einsum"""
    # TODO
    pass

def batch_trace(S):
    """(B, N, N) -> (B,) — sum of diagonal elements per matrix, no loop"""
    # TODO: think about how to extract diagonals across a batch
    pass

def batch_frobenius(A):
    """(B, N, M) -> (B,) Frobenius norm per matrix, no loop"""
    # Frobenius norm = sqrt(sum of squared elements)
    # TODO: reduce over last two axes
    pass

def scaled_dot_product_attention(Q, K, mask=None):
    """
    Q, K: (B, T, d_k)
    Returns attention weights: (B, T, T) — each row is a probability distribution
    mask: optional (T, T) boolean — True = masked (set to -inf before softmax)
    """
    # TODO: scores = Q @ Kᵀ / sqrt(d_k)
    # TODO: apply mask if provided
    # TODO: softmax over last axis
    pass

In [ ]:
# --- ASSERTS ---

# Batch matmul
C = batch_matmul(A_batch, B_batch)
assert C.shape == (B_size, N_size, K_size)
# Verify against manual loop
C_ref = np.stack([A_batch[i] @ B_batch[i] for i in range(B_size)])
assert np.allclose(C, C_ref, atol=1e-10), 'batch_matmul result mismatch'

# Batch trace
traces = batch_trace(S_batch)
assert traces.shape == (B_size,)
traces_ref = np.array([np.trace(S_batch[i]) for i in range(B_size)])
assert np.allclose(traces, traces_ref), 'batch_trace mismatch'

# Batch Frobenius
fnorms = batch_frobenius(A_batch)
assert fnorms.shape == (B_size,)
fnorms_ref = np.array([np.linalg.norm(A_batch[i], 'fro') for i in range(B_size)])
assert np.allclose(fnorms, fnorms_ref), 'batch_frobenius mismatch'

# Attention weights
attn = scaled_dot_product_attention(Q_attn, K_attn)
assert attn.shape == (B_size, T, T), f'Attention shape wrong: {attn.shape}'
assert np.allclose(attn.sum(axis=-1), 1.0, atol=1e-5), 'Attention weights should sum to 1'
assert np.all(attn >= 0), 'Negative attention weights'

# Causal mask test
causal_mask = np.triu(np.ones((T, T), dtype=bool), k=1)  # upper triangular = future
attn_masked = scaled_dot_product_attention(Q_attn, K_attn, mask=causal_mask)
# Future positions should have ~0 weight
assert np.allclose(attn_masked[0][np.triu_indices(T, k=1)], 0, atol=1e-6), 'Causal mask not applied'

print('P10 PASSED ✓')

---
## P11 — SGD with Momentum

Implement one step of SGD with momentum. The update rule:
```
v = momentum * v - lr * grad
W = W + v
```
Note: some implementations use `v = momentum*v + grad` then `W -= lr*v` — know both.

In [ ]:
def sgd_momentum_step(W, dW, v, lr=0.01, momentum=0.9):
    """
    W:        parameter array, any shape
    dW:       gradient, same shape as W
    v:        velocity, same shape as W
    Returns:  new_W, new_v  (do NOT modify in-place)
    """
    # TODO
    pass

# Test on a simple quadratic: f(W) = ||W||^2, grad = 2W
W0 = np.array([3.0, -2.0, 1.0])
v0 = np.zeros_like(W0)

W, v = W0.copy(), v0.copy()
history = [W.copy()]
for _ in range(20):
    grad = 2 * W  # gradient of ||W||^2
    W, v = sgd_momentum_step(W, grad, v, lr=0.1, momentum=0.9)
    history.append(W.copy())
history = np.array(history)

In [ ]:
# --- ASSERTS ---
W_init = np.array([3.0, -2.0, 1.0])
v_init = np.zeros(3)
W_new, v_new = sgd_momentum_step(W_init, 2*W_init, v_init, lr=0.01, momentum=0.9)

assert W_new.shape == W_init.shape
assert v_new.shape == v_init.shape

# Velocity after first step (from zero): v = -lr * grad
expected_v = -0.01 * 2 * W_init
assert np.allclose(v_new, expected_v), f'First-step velocity wrong: {v_new} vs {expected_v}'

# Convergence: after 20 steps on ||W||^2, W should be near 0
assert np.linalg.norm(history[-1]) < 0.1, f'Should converge to 0, got norm={np.linalg.norm(history[-1]):.4f}'

print('P11 PASSED ✓')

plt.figure(figsize=(6, 4))
for dim in range(3):
    plt.plot(history[:, dim], label=f'W[{dim}]')
plt.axhline(0, color='k', linestyle='--', linewidth=0.8)
plt.title('SGD+Momentum on ||W||² — convergence')
plt.xlabel('Step')
plt.legend()
plt.tight_layout()
plt.show()

---
## P12 — Adam Optimizer Step

Adam update rule:
```
m = beta1 * m + (1 - beta1) * grad          # first moment
v = beta2 * v + (1 - beta2) * grad²         # second moment
m_hat = m / (1 - beta1^t)                   # bias correction
v_hat = v / (1 - beta2^t)
W = W - lr * m_hat / (sqrt(v_hat) + eps)
```

In [ ]:
def adam_step(W, dW, m, v, t, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
    """
    W, dW, m, v: same shape
    t: current timestep (int, starts at 1)
    Returns: new_W, new_m, new_v
    """
    # TODO
    pass

# Compare convergence: Adam vs SGD on ||W||^2 + noisy gradients
W_sgd  = np.array([5.0, -3.0, 2.0, -4.0])
W_adam = W_sgd.copy()
v_sgd  = np.zeros_like(W_sgd)
m_adam = np.zeros_like(W_adam)
v_adam = np.zeros_like(W_adam)

sgd_hist, adam_hist = [W_sgd.copy()], [W_adam.copy()]
np.random.seed(7)

for t in range(1, 101):
    noise = np.random.randn(*W_sgd.shape) * 2.0
    grad_sgd  = 2 * W_sgd  + noise
    grad_adam = 2 * W_adam + noise
    
    W_sgd, v_sgd = sgd_momentum_step(W_sgd, grad_sgd, v_sgd, lr=0.05)
    W_adam, m_adam, v_adam = adam_step(W_adam, grad_adam, m_adam, v_adam, t)
    
    sgd_hist.append(W_sgd.copy())
    adam_hist.append(W_adam.copy())

sgd_hist  = np.array(sgd_hist)
adam_hist = np.array(adam_hist)

In [ ]:
# --- ASSERTS ---
W_t = np.ones(4)
m_t, v_t = np.zeros(4), np.zeros(4)
dW_t = np.ones(4)  # constant gradient

W_new, m_new, v_new = adam_step(W_t, dW_t, m_t, v_t, t=1)

# After t=1 with zero init:
# m = (1-0.9)*1 = 0.1, m_hat = 0.1/(1-0.9) = 1.0
# v = (1-0.999)*1 = 0.001, v_hat = 0.001/(1-0.999) = 1.0
# update = lr * 1.0 / (sqrt(1.0) + eps) ≈ lr
expected_W = W_t - 1e-3 * 1.0 / (1.0 + 1e-8)
assert np.allclose(W_new, expected_W, atol=1e-6), f'Adam first step wrong: {W_new} vs {expected_W}'

assert m_new.shape == W_t.shape
assert v_new.shape == W_t.shape
assert np.all(v_new >= 0), 'Second moment must be non-negative'

# Adam should converge under noise
assert np.linalg.norm(adam_hist[-1]) < np.linalg.norm(adam_hist[0]), 'Adam should reduce ||W||'

print('P12 PASSED ✓')

# Convergence comparison
sgd_norms  = np.linalg.norm(sgd_hist, axis=1)
adam_norms = np.linalg.norm(adam_hist, axis=1)

plt.figure(figsize=(7, 4))
plt.plot(sgd_norms, label='SGD+Momentum', linewidth=2)
plt.plot(adam_norms, label='Adam', linewidth=2)
plt.xlabel('Step')
plt.ylabel('||W||')
plt.title('SGD vs Adam — noisy gradients, ||W||² objective')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()